IMPORTING THE LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import requests, zipfile, io
import os

LOAD THE DATASET-SMS SPAM COLLECTION (UCI)

In [8]:
# Download and extract dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()

# Load into pandas DataFrame
SMS_df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "text"])

# Preview
print(SMS_df.head())


  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


LOAD THE DATASET-BBC NEWS DATASET (KAGGLE)

In [ ]:
import os
import pandas as pd
data = []
base_path = r"BBC News Summary\News Articles"  # folder path

for category in os.listdir(base_path):
    category_path = os.path.join(base_path, category)
    if os.path.isdir(category_path):
        for filename in os.listdir(category_path):
            file_path = os.path.join(category_path, filename)
            with open(file_path, "r", encoding="latin-1") as f:
                text = f.read()
                data.append((category, text))

bbc_df = pd.DataFrame(data, columns=["label", "text"])
print(bbc_df.head())
print("Total articles:", len(bbc_df))


# PREPROCESSING TEXT

PREPROCESS THE SMS SPAM COLLECTION

In [ ]:
def preprocess_text(text):

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

SMS_df['clean_text'] = SMS_df['text'].apply(preprocess_text)
SMS_df['tokens'] = SMS_df['clean_text'].apply(lambda x: x.split())

print(SMS_df[['label', 'tokens']].head())

PREPROCESS THE BBC NEWS DATASET

In [ ]:
bbc_df['clean_text'] = bbc_df['text'].apply(preprocess_text)
bbc_df['tokens'] = bbc_df['clean_text'].apply(lambda x: x.split())
print(bbc_df[['label', 'tokens']].head())

# BUILDING A VOCABULARY FROM TRAINING DATA

SMS SPAM COLLECTION SPLIT

In [ ]:
SMS_train_df, SMS_test_df = train_test_split(SMS_df, test_size=0.2, random_state=42)
print(f"SMS Train Size: {len(SMS_train_df)}")
print(f"SMS Test Size: {len(SMS_test_df)}")

BBC NEWS DATASET SPLIT

In [ ]:
BBC_train_df, BBC_test_df = train_test_split(bbc_df, test_size=0.2, random_state=42, stratify=bbc_df['label'])
print(f"BBC Train Size: {len(BBC_train_df)}")
print(f"BBC Test Size: {len(BBC_test_df)}")

In [ ]:
def build_vocabulary(df):
    """Builds a set of all unique words from the 'tokens' column."""
    vocab = set()
    for doc_tokens in df['tokens']:
        vocab.update(doc_tokens)
    return vocab

# Build vocabulary for each dataset
SMS_VOCAB = build_vocabulary(SMS_train_df)
BBC_VOCAB = build_vocabulary(BBC_train_df)

print(f"SMS Vocabulary Size: {len(SMS_VOCAB)}")
print(f"BBC Vocabulary Size: {len(BBC_VOCAB)}")

# NAIVE BAYES CLASSIFIER CLASS

THIS CLASS WILL IMPLEMENT THE LOGIC TO COMPUTE PRIORS,LIKELIHOODS AND POSTERIORS SCORES 

In [ ]:
class NaiveBayesClassifier:
    def __init__(self, alpha=1.0, model_type='multinomial'):
        self.alpha = alpha  # Laplace smoothing parameter [cite: 48]
        self.model_type = model_type
        self.P_C = {}       # Prior probabilities P(C) [cite: 31]
        self.P_wC = {}      # Likelihoods P(w|C) [cite: 33]
        self.classes = []
        self.vocab = set()

    def train(self, df):
        """Trains the Naive Bayes model based on the chosen likelihood type."""
        self.classes = df['label'].unique()
        self.vocab = build_vocabulary(df)
        V = len(self.vocab)
        
        # 1. Compute Prior Probabilities P(C) [cite: 107]
        total_docs = len(df)
        for C in self.classes:
            C_docs = df[df['label'] == C]
            self.P_C[C] = len(C_docs) / total_docs  # P(C) = #docs in C / total #docs [cite: 32]

            # 2. Compute Likelihoods P(w|C) [cite: 108]
            if self.model_type == 'multinomial':
                # Multinomial (Count-Based) Likelihood [cite: 41]
                word_counts = Counter()
                for tokens in C_docs['tokens']:
                    word_counts.update(tokens)
                
                total_words_in_C = sum(word_counts.values())
                
                self.P_wC[C] = {}
                for w in self.vocab:
                    # P(w|C) = (count of w in C + alpha) / (total words in C + alpha*|V|) [cite: 45, 46]
                    self.P_wC[C][w] = (word_counts.get(w, 0) + self.alpha) / (total_words_in_C + self.alpha * V)
            
            elif self.model_type == 'binary':
                # Binary ON/OFF (Presence-Based) Likelihood [cite: 54]
                docs_in_C = len(C_docs)
                self.P_wC[C] = {}
                
                for w in self.vocab:
                    # Count documents in C containing word w
                    docs_with_w = sum(w in doc_tokens for doc_tokens in C_docs['tokens'])
                    
                    # P(w|C) = (#docs in C containing w + alpha) / (#docs in C + 2*alpha) [cite: 58]
                    self.P_wC[C][w] = (docs_with_w + self.alpha) / (docs_in_C + 2 * self.alpha)

    def predict_doc(self, tokens):
        """Computes the posterior score for each class and returns the prediction."""
        best_score = -np.inf
        predicted_class = None
        
        # Unique words in the document for binary model tracking
        doc_unique_words = set(tokens)
        
        # Word counts in the document for multinomial model
        doc_word_counts = Counter(tokens)
        
        # 3. Compute Posterior Scores log P(C|w_d) [cite: 109]
        for C in self.classes:
            # score(C, w_d) = log P(C) + sum(log P(w|C)) [cite: 72]
            score = np.log(self.P_C[C]) # Prior term log P(C) [cite: 73]
            
            if self.model_type == 'multinomial':
                # Sum of log likelihoods, weighted by word counts (Multinomial) [cite: 52]
                for w, count in doc_word_counts.items():
                    if w in self.vocab:
                        score += count * np.log(self.P_wC[C][w])
                        
            elif self.model_type == 'binary':
                # Sum of log likelihoods for presence/absence (Binary ON/OFF) [cite: 64]
                for w in self.vocab:
                    if w in doc_unique_words:
                        # Word is present: use log P(w|C)
                        score += np.log(self.P_wC[C][w])
                    else:
                        # Word is absent: use log (1 - P(w|C)) [cite: 60, 64]
                        score += np.log(1.0 - self.P_wC[C][w])

            if score > best_score:
                best_score = score
                predicted_class = C
                
        return predicted_class

    def predict(self, df):
        """Applies prediction to a DataFrame of test documents."""
        return [self.predict_doc(tokens) for tokens in df['tokens']]

    def evaluate(self, df):
        """Computes and returns the accuracy of the predictions."""
        predictions = self.predict(df)
        accuracy = np.mean(predictions == df['label'])
        return accuracy, predictions

# TRAINING AND EVALUATION

1) SMS SPAM CLASSIFICATION (BINARY ON/OFF MODEL)

In [ ]:
# Train with Binary ON/OFF Model (Alpha=1 is the default for Laplace smoothing)
sms_classifier = NaiveBayesClassifier(alpha=1.0, model_type='binary')
sms_classifier.train(SMS_train_df)

# Test and compute accuracy
sms_accuracy, sms_predictions = sms_classifier.evaluate(SMS_test_df)
print(f"SMS Spam Classifier Test Accuracy (Binary Model): {sms_accuracy:.4f}")

# Verify Test Cases [cite: 116]
test_messages = [
    "Win free tickets now!!!",        # Expected: spam [cite: 117]
    "Are you coming to the meeting?",  # Expected: ham [cite: 117]
    "URGENT! You won $1000",           # Expected: spam [cite: 117]
    "See you tomorrow at lunch"        # Expected: ham [cite: 117]
]
test_tokens = [preprocess_text(msg).split() for msg in test_messages]
test_results = [sms_classifier.predict_doc(t) for t in test_tokens]

print("\nSMS Test Case Verification:")
for i, msg in enumerate(test_messages):
    print(f"Message: '{msg}' -> Predicted: {test_results[i]}")

2. BBC NEWS CLASSIFICATION (MULTINOMIAL MODEL)

In [ ]:
from sklearn.metrics import confusion_matrix
# Train with Multinomial (Count-Based) Model
bbc_classifier = NaiveBayesClassifier(alpha=1.0, model_type='multinomial')
bbc_classifier.train(BBC_train_df)

# Test and compute accuracy
bbc_accuracy, bbc_predictions = bbc_classifier.evaluate(BBC_test_df)
print(f"\nBBC News Classifier Test Accuracy (Multinomial Model): {bbc_accuracy:.4f}")

# Compute Confusion Matrix (Assignment Task 6)
C = confusion_matrix(BBC_test_df['label'], bbc_predictions, labels=bbc_classifier.classes)
cm_df = pd.DataFrame(C, index=bbc_classifier.classes, columns=bbc_classifier.classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix for BBC News')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.show()

# Verify Test Cases [cite: 118]
bbc_test_snippets = [
    "Stock market crashes as oil prices rise",          # Expected: business [cite: 119]
    "Premier League team wins the championship",        # Expected: sport [cite: 119]
    "Government passes new healthcare reform",          # Expected: politics [cite: 119]
    "Apple releases latest iPhone with new features",   # Expected: tech [cite: 119]
    "Celebrity announces new film project"              # Expected: entertainment [cite: 119]
]
bbc_test_tokens = [preprocess_text(snip).split() for snip in bbc_test_snippets]
bbc_test_results = [bbc_classifier.predict_doc(t) for t in bbc_test_tokens]

print("\nBBC Test Case Verification:")
for i, snip in enumerate(bbc_test_snippets):
    print(f"Snippet: '{snip}' -> Predicted: {bbc_test_results[i]}")

# ANALYSIS: TOP 10 INDICATIVE WORDS

To find the words most indicative of a class C, we can look at the ratio of the class likelihood P(w∣C) to the likelihood of the word appearing in all other classes. A simple and effective approach is to find the words with the highest logP(w∣C) since we're using logs for classification.

Let's modify the analysis to look at the ratio of likelihoods to provide a stronger measure of indicativeness (how much more likely a word is to appear in class C than in any other class).

In [ ]:
def get_top_indicative_words(classifier, N=10):
    """Finds the top N words that are most indicative of each class."""
    indicative_words = defaultdict(list)
    
    for C in classifier.classes:
        # Calculate the log-ratio: log(P(w|C)) - log(P(w|Not C))
        # P(w|Not C) is estimated as the average P(w|C') for all C' != C
        word_scores = {}
        for w in classifier.vocab:
            log_p_wC = np.log(classifier.P_wC[C].get(w, 1e-10)) # use a tiny value for safety
            
            # Calculate log P(w|Not C)
            log_p_wNotC_list = []
            for C_prime in classifier.classes:
                if C_prime != C:
                    log_p_wNotC_list.append(np.log(classifier.P_wC[C_prime].get(w, 1e-10)))
            
            # Use the mean of log P(w|C') as a proxy for log P(w|Not C)
            log_p_wNotC = np.mean(log_p_wNotC_list)
            
            score = log_p_wC - log_p_wNotC
            word_scores[w] = score

        # Get top N words by score
        sorted_words = sorted(word_scores.items(), key=lambda item: item[1], reverse=True)
        indicative_words[C] = sorted_words[:N]
        
    return indicative_words

# 1. SMS Spam Analysis
sms_indicative = get_top_indicative_words(sms_classifier)
print("\n--- SMS Spam Top 10 Indicative Words ---")
for C, words in sms_indicative.items():
    print(f"\nClass '{C}':")
    print(", ".join([f"{w}" for w, score in words]))

# 2. BBC News Analysis
bbc_indicative = get_top_indicative_words(bbc_classifier)
print("\n--- BBC News Top 10 Indicative Words ---")
for C, words in bbc_indicative.items():
    print(f"\nClass '{C}':")
    print(", ".join([f"{w}" for w, score in words]))